# [9.4] White-box Evals and Monitors - Exercises

Implement the local monitor report primitives, then run the visible tests in each cell. The CUDA verification report for the full Pythia preflight is committed next to this notebook.

In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter9_alignment_interpretability"
section = "part4_white_box_evals_monitors"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_white_box_evals_monitors.tests as tests
import part4_white_box_evals_monitors.utils as utils

GT_TIER = "GT-3"
EXERCISE_ID = "9.4.white_box_evals_and_monitors"
EXPECTED_RUNTIME = "35-50 minutes for exercises; about 1-3 minutes for the CUDA preflight"
REQUIRES_GPU = True

In [ ]:
@dataclass(frozen=True)
class MonitorDashboardRow:
    prompt: str
    model_output: str
    active_features: tuple[str, ...]
    refusal_score: float
    hallucination_score: float
    cot_faithfulness_score: float


@dataclass(frozen=True)
class MonitorCalibrationReport:
    auroc: float
    calibrated: bool


@dataclass(frozen=True)
class MissedFailureReport:
    caught_failure_indices: tuple[int, ...]
    num_caught_failures: int
    catches_black_box_miss: bool


@dataclass(frozen=True)
class FalsePositiveDocumentationReport:
    false_positive_indices: tuple[int, ...]
    num_false_positives: int
    documented: bool


@dataclass(frozen=True)
class FeatureExplanationValidationReport:
    heldout_accuracy: float
    explanations_validated: bool

## Dashboard Rows

In [ ]:
def monitor_dashboard_row(
    *,
    prompt: str,
    model_output: str,
    active_features: list[str],
    refusal_score: float,
    hallucination_score: float,
    cot_faithfulness_score: float,
) -> MonitorDashboardRow:
    raise NotImplementedError()


tests.test_monitor_dashboard_row_preserves_review_fields(monitor_dashboard_row)

## Calibration

In [ ]:
def binary_auroc(scores: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


tests.test_binary_auroc_counts_ties_and_validates_inputs(binary_auroc)

In [ ]:
def monitor_calibration_report(
    monitor_scores: t.Tensor,
    failure_labels: t.Tensor,
    *,
    min_auroc: float = 0.8,
) -> MonitorCalibrationReport:
    raise NotImplementedError()


tests.test_monitor_calibration_report_matches_reference(monitor_calibration_report)

## White-box Versus Black-box

In [ ]:
def missed_failure_report(
    white_box_predictions: t.Tensor,
    black_box_predictions: t.Tensor,
    failure_labels: t.Tensor,
) -> MissedFailureReport:
    raise NotImplementedError()


tests.test_missed_failure_report_identifies_white_box_only_catches(missed_failure_report)

## False Positives And Explanations

In [ ]:
def false_positive_documentation_report(
    monitor_predictions: t.Tensor,
    failure_labels: t.Tensor,
    documentation: dict[int, str] | None = None,
) -> FalsePositiveDocumentationReport:
    raise NotImplementedError()


tests.test_false_positive_documentation_requires_notes(
    false_positive_documentation_report,
)

In [ ]:
def feature_explanation_validation_report(
    explanation_predictions: t.Tensor,
    heldout_labels: t.Tensor,
    *,
    min_accuracy: float = 0.8,
) -> FeatureExplanationValidationReport:
    raise NotImplementedError()


tests.test_feature_explanation_validation_uses_heldout_accuracy(
    feature_explanation_validation_report,
)

## Smoke Contract

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "dashboard": monitor_dashboard_row(
            prompt="Summarize this harmless note.",
            model_output="A concise summary.",
            active_features=["summary", "benign"],
            refusal_score=0.1,
            hallucination_score=0.2,
            cot_faithfulness_score=0.9,
        ).__dict__,
        "calibration": monitor_calibration_report(
            t.tensor([0.1, 0.4, 0.8, 0.9]),
            t.tensor([0, 0, 1, 1], dtype=t.bool),
            min_auroc=0.9,
        ).__dict__,
        "missed_failure": missed_failure_report(
            t.tensor([1, 0, 0], dtype=t.bool),
            t.tensor([0, 0, 0], dtype=t.bool),
            t.tensor([1, 1, 0], dtype=t.bool),
        ).__dict__,
        "false_positive": false_positive_documentation_report(
            t.tensor([1, 0, 1], dtype=t.bool),
            t.tensor([1, 0, 0], dtype=t.bool),
            documentation={2: "Benign style feature caused a high monitor score."},
        ).__dict__,
        "explanation_validation": feature_explanation_validation_report(
            t.tensor([1, 0, 1, 0], dtype=t.bool),
            t.tensor([1, 0, 1, 0], dtype=t.bool),
            min_accuracy=1.0,
        ).__dict__,
    }


tests.test_notebook_contract(run_smoke_test)


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    gpu = report["metrics"]["gpu_test"]
    tests.test_committed_gpu_report_matches_white_box_monitor_contract(gpu)
    assert report["accepted"] and report["tests_passed"]
    assert report["gt_tier"] == "GT-3"
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
{key: gpu[key] for key in [
    "preflight_passed",
    "monitor_auroc",
    "white_box_accuracy",
    "black_box_proxy_accuracy",
    "catches_black_box_miss",
    "false_positives_documented",
    "explanations_validated",
    "peak_vram_gb",
]}
